In [1]:
print("Hello")

Hello


In [2]:
!uv add langchain langchain-community google-generativeai langchain-google-genai rapidocr-onnxruntime python-dotenv

Resolved 123 packages in 1ms
Audited 119 packages in 4ms


In [3]:
!uv pip install langchain langchain-community google-generativeai langchain-google-genai rapidocr-onnxruntime python-dotenv

Using Python 3.14.2 environment at: D:\Projects\DOC_CHAT\.venv
Audited 6 packages in 10ms


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

# os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

True

## Data Ingestion

In [5]:
!uv pip install pypdf

Using Python 3.14.2 environment at: D:\Projects\DOC_CHAT\.venv
Audited 1 package in 3ms


In [6]:
!uv add langchain-text-splitters

Resolved 123 packages in 0.87ms
Audited 119 packages in 2ms


In [2]:
import os
import time
import hashlib
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

# from sentence_transformers import SentenceTransformer
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

from ragas.metrics import DiscreteMetric 

d:\Projects\DOC_CHAT\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
d:\Projects\DOC_CHAT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Projects\DOC_CHAT\.venv\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


In [2]:
def process_pdf(file_path: str) -> list[str]:
    """Loads a PDF and splits it into manageable text chunks."""
   
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = splitter.split_documents(documents)
    
    
    return [chunk.page_content for chunk in chunks]

In [3]:
def generate_id(text: str) -> str:
    """Generates a unique MD5 hash for a given text chunk to prevent duplicates."""
    return hashlib.md5(text.encode('utf-8')).hexdigest()

In [4]:
def ingest_to_pinecone(file_path: str):
    """Orchestrates the chunking, embedding, and storage process."""
    
    # Chunk the PDF
    print(f"Processing {file_path}...")
    text_chunks = process_pdf(file_path)
    
    print("Loading Gemini embedding model...")
    embeddings_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", output_dimensionality=768)
    
    
    # rate-limit
    print("Embedding documents in batches...")
    embeddings = []
    embed_batch_size = 40
     
    
    for i in range(0, len(text_chunks), embed_batch_size):
        batch = text_chunks[i : i + embed_batch_size]
        print(f"Embedding batch {i // embed_batch_size + 1}...")
        
        try:
            batch_embeddings = embeddings_model.embed_documents(batch)
            embeddings.extend(batch_embeddings)
            time.sleep(15) 
        except Exception as e:
            print(f"Rate limit hit! Sleeping for 50 seconds... Error: {e}")
            time.sleep(50)
            batch_embeddings = embeddings_model.embed_documents(batch)
            embeddings.extend(batch_embeddings)
    
    print("Connecting to Pinecone...")
    pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
    
    # Create the index if it doesn't already exist
    if INDEX_NAME not in pc.list_indexes().names():
        print(f"Creating new index: {INDEX_NAME}...")
        pc.create_index(
            name=INDEX_NAME,
            dimension=768, # Must match the dimension output of your embedding model
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
    
    index = pc.Index(INDEX_NAME)
    
    print("Packaging data and upserting...")
    vectors_to_upsert = []
    
    for chunk, embedding in zip(text_chunks, embeddings):
        chunk_id = generate_id(chunk)
        
        record = {
            "id": chunk_id,
            "values": embedding,
            "metadata": {
                "text": chunk,
                "source_file": os.path.basename(file_path)
            }
        }
        vectors_to_upsert.append(record)
        
    # Best practice: Don't send 10,000 vectors in a single HTTP request. Batch them.
    batch_size = 100
    for i in range(0, len(vectors_to_upsert), batch_size):
        batch = vectors_to_upsert[i : i + batch_size]
        index.upsert(vectors=batch, namespace="default")
        time.sleep(5)
    print(f"Successfully upserted {len(vectors_to_upsert)} chunks to Pinecone!")

## Retriever

In [5]:
def retrieve_context(query: str, index_name: str = "doc-chat", top_k: int = 3):
    """
    Converts a user query into an embedding and retrieves the most relevant 
    text chunks from the Pinecone vector database.
    """
    
    embeddings_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", output_dimensionality=768)
    
    query_embedding = embeddings_model.embed_query(query)
    
    pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
    index = pc.Index(index_name)
    

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        namespace="default"
    )
    
    retrieved_chunks = [match["metadata"]["text"] for match in results["matches"]]
    
    return retrieved_chunks

## Generation

In [6]:
def generate_answer(query: str, context_chunks: list[str]):
    """
    Takes retrieved text chunks and uses Gemini to synthesize a 
    natural language response.
    """
    
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.environ.get("GEMINI_API_KEY"),
        temperature=0.1  # Low temperature for factual consistency
    )
    
    system_prompt = (
        "You are a helpful assistant. Use the following pieces of retrieved context "
        "to answer the user's question. Make sure to go through the context thoroughly"
        "If you don't know the answer based on the context,"
        "just say that you don't know. Don't try to make up an answer.\n\n"
        "Context:\n{context}"
    )
    
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}"),
    ])
    
    formatted_context = "\n\n---\n\n".join(context_chunks)
    
    chain = prompt_template | llm
    
    response = chain.invoke({
        "context": formatted_context,
        "question": query
    })
    
    return response.content

In [18]:
if __name__ == "__main__":
    
    user_query = "Who is the author of uploaded documnet?"
    INDEX_NAME = 'doc-chat'
    file_path = r'D:\Projects\DOC_CHAT\data\Data Security & Compliance Framework Final - Parth(Corrected).pdf'
    
    # ingest_to_pinecone(file_path)

    # Step 1: Retrieve
    retrieved_snippets = retrieve_context(user_query) 
    
    # Step 2: Generate
    final_answer = generate_answer(user_query, retrieved_snippets)
    
    print(f"Question: {user_query}")
    print(f"\nAI Answer:\n{final_answer}")

Question: Who is the author of uploaded documnet?

AI Answer:
I don't know the answer based on the context. The provided document snippets do not mention the author.


In [17]:
user_query_1 = "How many policies are there in a document?" 
retrieved_snippets = retrieve_context(user_query_1)
generate_answer(user_query_1, retrieved_snippets)

'There are 12 mandatory policies in the document.'